In [0]:
from pyspark.sql import functions as F
CATALOG, SCHEMA = "workspace", "olist"
TS = "yyyy-MM-dd HH:mm:ss" 

def bronze(n): return spark.table(f"{CATALOG}.{SCHEMA}.bronze_{n}")
def silver(n): return spark.table(f"{CATALOG}.{SCHEMA}.silver_{n}")

def dedupe(df, keys, name):
    out = df.dropDuplicates(keys)
    print(f"{name}: {df.count()} -> {out.count()} rows")
    return out

def save_silver(df, n):
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true")
       .saveAsTable(f"{CATALOG}.{SCHEMA}.silver_{n}"))
    print("saved silver_" + n, spark.table(f"{CATALOG}.{SCHEMA}.silver_{n}").count(), "rows")

In [0]:
# ORDERS: text -> timestamps, tidy status, one row per order_id
orders = bronze("orders")
for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    orders = orders.withColumn(c, F.to_timestamp(c, TS))
orders = orders.withColumn("order_status", F.lower(F.trim("order_status")))
orders = dedupe(orders, ["order_id"], "orders").drop("_ingested_at")

# ORDER ITEMS: numbers as decimals (exact money), one row per (order_id, order_item_id)
items = (bronze("order_items")
         .withColumn("order_item_id", F.col("order_item_id").cast("int"))
         .withColumn("shipping_limit_date", F.to_timestamp("shipping_limit_date", TS))
         .withColumn("price", F.col("price").cast("decimal(12,2)"))
         .withColumn("freight_value", F.col("freight_value").cast("decimal(12,2)")))
items = dedupe(items, ["order_id", "order_item_id"], "order_items").drop("_ingested_at")

# CUSTOMERS: tidy text, one row per customer_id
customers = (bronze("customers")
             .withColumn("customer_state", F.upper(F.trim("customer_state")))
             .withColumn("customer_city", F.lower(F.trim("customer_city"))))
customers = dedupe(customers, ["customer_id"], "customers").drop("_ingested_at")

# PAYMENTS: numbers as numbers, one row per (order_id, payment_sequential)
payments = (bronze("payments")
            .withColumn("payment_sequential", F.col("payment_sequential").cast("int"))
            .withColumn("payment_installments", F.col("payment_installments").cast("int"))
            .withColumn("payment_value", F.col("payment_value").cast("decimal(12,2)")))
payments = dedupe(payments, ["order_id", "payment_sequential"], "payments").drop("_ingested_at")

# PRODUCTS: fix the dataset's misspelled columns, cast sizes to int, add English category (same COALESCE rule as your SQL)
trans = bronze("category_translation").dropDuplicates(["product_category_name"]) \
        .select("product_category_name", "product_category_name_english")
products = (bronze("products")
            .withColumnRenamed("product_name_lenght", "product_name_length")
            .withColumnRenamed("product_description_lenght", "product_description_length"))
for c in ["product_name_length", "product_description_length", "product_photos_qty",
          "product_weight_g", "product_length_cm", "product_height_cm", "product_width_cm"]:
    products = products.withColumn(c, F.col(c).cast("int"))
products = dedupe(products, ["product_id"], "products")
products = (products.join(trans, "product_category_name", "left")
            .withColumn("category", F.coalesce("product_category_name_english", "product_category_name", F.lit("unknown")))
            .drop("_ingested_at"))

for df, n in [(orders, "orders"), (items, "order_items"), (customers, "customers"),
              (payments, "payments"), (products, "products")]:
    save_silver(df, n)

orders: 99441 -> 99441 rows
order_items: 112650 -> 112650 rows
customers: 99441 -> 99441 rows
payments: 103886 -> 103886 rows
products: 32951 -> 32951 rows
saved silver_orders 99441 rows
saved silver_order_items 112650 rows
saved silver_customers 99441 rows
saved silver_payments 103886 rows
saved silver_products 32951 rows


In [0]:
sales = (silver("order_items")
         .join(silver("orders"), "order_id", "left")
         .join(silver("customers").select("customer_id", "customer_unique_id", "customer_city", "customer_state"),
               "customer_id", "left")
         .join(silver("products").select("product_id", "category"), "product_id", "left")
         .withColumn("category", F.coalesce("category", F.lit("unknown"))))
save_silver(sales, "sales")

print("items:", silver("order_items").count(), "| sales:", silver("sales").count())
rev = silver("sales").filter("order_status = 'delivered'").agg(F.round(F.sum("price"), 2)).first()[0]
print("delivered revenue:", rev, "| SQLite checkpoint: 13221498.11")

saved silver_sales 112650 rows
items: 112650 | sales: 112650
delivered revenue: 13221498.11 | SQLite checkpoint: 13221498.11
